# ML2 Final Project

## Set up
1. Change Runtime type to T4 GPU

In [4]:
import torch
import os
from google.colab import drive
import zipfile
from tensorflow.keras.applications import VGG16 # teacher model
import pandas as pd
!pip install deepface
from deepface import DeepFace
from deepface.models.facial_recognition.VGGFace import VggFaceClient
from tensorflow.keras import Model
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator



# make sure this doesn't raise an error
assert(torch.cuda.is_available())

In [14]:
# mount this notebook to Google Drive (run this only once for ease)
drive.mount('/content/drive')

# path to the team folder and the data file (.zip)
data_path = '/content/drive/Shareddrives/TEAM 7 Machine Learning II/Final Project/data.zip'

# once we unzip the data
images_path = '/content/data/BMI/Data/Images'
csv_path = '/content/data/BMI/Data/data.csv'

In [16]:
# unzip the data file and save to /content (need to do this every session since Colab's local disk resets when the runtime disconnects)
with zipfile.ZipFile(data_path, 'r') as z:
    z.extractall('/content/data') # unzips everything into Colab local disk (RAM-backed, fast)

# this confirms the images are loaded into content
print(len(os.listdir(images_path))) # 3,963 images

print(os.path.exists(csv_path))

3963
True


## Load VGG16 from keras (built into tensorflow).

Not identical to VGG-Face, but pretty close.

VGG16 is a CNN trained on ImageNet to classify 1,000 objects. The last few layers are:
- conv layers to detect edges, textures, shapes
- avg pooling (512 numbers to summarize what the network sees)
- final classification layer, but we remove that using the `include_top=False` param)

We can use VGG16 to fine-tune starts from the pretrained ImageNet weights
Then we'll swap the final layer (which outputs 1k class probabilities) with a single output (BMI value).

In [20]:
model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3), pooling='avg')

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


## Inference check/test using the VGG16 model

In [22]:
import numpy as np
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.preprocessing import image

# load one image
img = image.load_img('/content/data/BMI/Data/Images/img_0.bmp', target_size=(224, 224))
x = image.img_to_array(img)
x = np.expand_dims(x, axis=0)
x = preprocess_input(x)

features = model.predict(x)
print(features.shape)  # should be (1, 512)

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
(1, 512)


## Load and prep the training data

*   Read in the csv
* Add image paths to each row in the .csv
* split into train and test sets


In [24]:
df = pd.read_csv(csv_path)
del df['Unnamed: 0']
display(df.head())

,bmi,gender,is_training,name
0,34.207396,Male,1,img_0.bmp
1,26.453720,Male,1,img_1.bmp
2,34.967561,Female,1,img_2.bmp
3,22.044766,Female,1,img_3.bmp
4,37.758789,Female,1,img_4.bmp


In [28]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import pandas as pd

# add full image path to df
df['path'] = images_path + '/' + df['name']

# filter to only existing images
df = df[df['path'].apply(os.path.exists)].reset_index(drop=True)

# train/test split using is_training column
train_df = df[df['is_training'] == 1].reset_index(drop=True)
test_df = df[df['is_training'] == 0].reset_index(drop=True)

print(train_df.shape, test_df.shape) # should closely match the paper's 3,368 vs 838 split (although not 100% because we have some missing images)

(3210, 5) (752, 5)


## Build generators

Get a method to stream 32 images at a time during training (and loads, preprocesses, and feeds to model). Also flips, rotates and zooms on images to increase data variety to avoid overfitting

Note: test generator does _not_ augment (so we can evaluate on unmodified images)

In [29]:
# augment training data to help generalization
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    horizontal_flip=True,
    rotation_range=10,
    zoom_range=0.1
)

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_dataframe(
    train_df, x_col='path', y_col='bmi',
    target_size=(224, 224), batch_size=32,
    class_mode='raw'
)

test_gen = test_datagen.flow_from_dataframe(
    test_df, x_col='path', y_col='bmi',
    target_size=(224, 224), batch_size=32,
    class_mode='raw', shuffle=False
)

Found 3210 validated image filenames.
Found 752 validated image filenames.


# Design new model (using VGG16 as a base)
# (define the architecture for the neural net)

In [30]:
# freeze all VGG16 layers first
for layer in model.layers:
    layer.trainable = False

# unfreeze last 4 layers for fine-tuning
for layer in model.layers[-4:]:
    layer.trainable = True

# add regression head
x = model.output
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='linear')(x)  # single BMI value

bmi_model = Model(inputs=model.input, outputs=output)
bmi_model.compile(optimizer=Adam(1e-4), loss='mse', metrics=['mae'])

# Train the model

Will take some time (T4 + 3,210 images + 30 epochs = ~20 min; however, at some point, early stopping will kick in)

* early stopping kicks in when validation loss stops improving for 5 epochs in a row (`patience=5`). Can increase this param value to train it longer before giving up, or lower to stop sooner.

In [31]:
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ModelCheckpoint('best_model.h5', save_best_only=True)
]

history = bmi_model.fit(
    train_gen,
    validation_data=test_gen,
    epochs=30,
    callbacks=callbacks
)

Epoch 1/30
101/101 ━━━━━━━━━━━━━━━━━━━━ 0s 523ms/step - loss: 208.2481 - mae: 10.9604

101/101 ━━━━━━━━━━━━━━━━━━━━ 84s 650ms/step - loss: 135.5736 - mae: 8.8978 - val_loss: 95.5391 - val_mae: 7.2793
Epoch 2/30
101/101 ━━━━━━━━━━━━━━━━━━━━ 0s 430ms/step - loss: 76.0121 - mae: 6.7412

101/101 ━━━━━━━━━━━━━━━━━━━━ 48s 474ms/step - loss: 81.1193 - mae: 6.9635 - val_loss: 93.1296 - val_mae: 7.4266
Epoch 3/30
101/101 ━━━━━━━━━━━━━━━━━━━━ 0s 435ms/step - loss: 73.2058 - mae: 6.6072

101/101 ━━━━━━━━━━━━━━━━━━━━ 49s 477ms/step - loss: 69.5420 - mae: 6.4078 - val_loss: 77.7537 - val_mae: 6.3534
Epoch 4/30
101/101 ━━━━━━━━━━━━━━━━━━━━ 82s 479ms/step - loss: 64.5858 - mae: 6.2133 - val_loss: 87.7990 - val_mae: 6.7668
Epoch 5/30
101/101 ━━━━━━━━━━━━━━━━━━━━ 49s 488ms/step - loss: 59.1897 - mae: 5.9365 - val_loss: 101.2642 - val_mae: 7.4445
Epoch 6/30
101/101 ━━━━━━━━━━━━━━━━━━━━ 48s 473ms/step - loss: 57.0204 - mae: 5.8443 - val_loss: 80.5161 - val_mae: 6.8336
Epoch 7/30
101/101 ━━━━━━━━━━━━━━━━━━━━ 0s 431ms/step - loss: 49.0766 - mae: 5.4515

101/101 ━━━━━━━━━━━━━━━━━━━━ 48s 473ms/step - loss: 50.6434 - mae: 5.4983 - val_loss: 76.8509 - val_mae: 6.3822
Epoch 8/30
101/101 ━━━━━━━━━━━━━━━━━━━━ 0s 436ms/step - loss: 48.4525 - mae: 5.3106

101/101 ━━━━━━━━━━━━━━━━━━━━ 50s 491ms/step - loss: 49.9493 - mae: 5.4585 - val_loss: 75.0652 - val_mae: 6.2269
Epoch 9/30
101/101 ━━━━━━━━━━━━━━━━━━━━ 0s 433ms/step - loss: 51.6872 - mae: 5.5142

101/101 ━━━━━━━━━━━━━━━━━━━━ 48s 475ms/step - loss: 47.3128 - mae: 5.3230 - val_loss: 69.7047 - val_mae: 6.0717
Epoch 10/30
101/101 ━━━━━━━━━━━━━━━━━━━━ 48s 470ms/step - loss: 43.3004 - mae: 5.0868 - val_loss: 78.5083 - val_mae: 6.3359
Epoch 11/30
101/101 ━━━━━━━━━━━━━━━━━━━━ 48s 469ms/step - loss: 39.8690 - mae: 4.8957 - val_loss: 81.0142 - val_mae: 6.4489
Epoch 12/30
101/101 ━━━━━━━━━━━━━━━━━━━━ 48s 473ms/step - loss: 36.6497 - mae: 4.7252 - val_loss: 72.2918 - val_mae: 6.1671
Epoch 13/30
101/101 ━━━━━━━━━━━━━━━━━━━━ 48s 473ms/step - loss: 37.2664 - mae: 4.7954 - val_loss: 73.5572 - val_mae: 6.2014
Epoch 14/30
101/101 ━━━━━━━━━━━━━━━━━━━━ 81s 466ms/step - loss: 34.6739 - mae: 4.5866 - val_loss: 71.3950 - val_mae: 6.1071


In [32]:
from scipy.stats import pearsonr
import numpy as np

preds = bmi_model.predict(test_gen).flatten()
actuals = test_df['bmi'].values

r, _ = pearsonr(preds, actuals)
print(f"Pearson r: {r:.4f}")

24/24 ━━━━━━━━━━━━━━━━━━━━ 7s 215ms/step
Pearson r: 0.4845


### Damn Pearson socre of 0.4845, way worse than the paper's 0.65. Probably because VGG16 is object-trained, not face-trained. Fuck that. Let's try something else.

### We need 3-4 models anyway, so this can be our "even-worse than baseline, but we had to start somewhere 🤷" model

# VGG Face

In [36]:
vggface_model = VggFaceClient()

In [37]:
# see summary, param counts, and name of each layer in VGG Face
# the last layer is 4,096 dimensions - same as the paper
vggface_keras = vggface_model.model
vggface_keras.summary()

Model: "functional_78"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zero_padding2d_13               │ (None, 226, 226, 3)    │             0 │
│ (ZeroPadding2D)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_16 (Conv2D)              │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zero_padding2d_14               │ (None, 226, 226, 64)   │             0 │
│ (ZeroPadding2D)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zero_padding2d_15               │ (None, 114, 114, 64)   │             0 │
│ (ZeroPadding2D)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_18 (Conv2D)              │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zero_padding2d_16               │ (None, 114, 114, 128)  │             0 │
│ (ZeroPadding2D)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_19 (Conv2D)              │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zero_padding2d_17               │ (None, 58, 58, 128)    │             0 │
│ (ZeroPadding2D)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_20 (Conv2D)              │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zero_padding2d_18               │ (None, 58, 58, 256)    │             0 │
│ (ZeroPadding2D)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_21 (Conv2D)              │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zero_padding2d_19               │ (None, 58, 58, 256)    │             0 │
│ (ZeroPadding2D)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_22 (Conv2D)              │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zero_padding2d_20               │ (None, 30, 30, 256)    │             0 │
│ (ZeroPadding2D)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_23 (Conv2D)              │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zero_padding2d_21               │ (None, 30, 30, 512)    │             

 Total params: 134,260,544 (512.16 MB)

 Trainable params: 134,260,544 (512.16 MB)

 Non-trainable params: 0 (0.00 B)

# Design new model (using VGG-Face as a base)
# (define the architecture for the neural net)

In [38]:
# freeze all layers
for layer in vggface_keras.layers:
    layer.trainable = False

# unfreeze last 4
for layer in vggface_keras.layers[-4:]:
    layer.trainable = True

# regression head on top of the 4096 flatten output
x = vggface_keras.output
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='linear')(x)

bmi_model_vggface = Model(inputs=vggface_keras.input, outputs=output)
bmi_model_vggface.compile(optimizer=Adam(1e-4), loss='mse', metrics=['mae'])

# And now more training

In [39]:
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ModelCheckpoint('best_model_vggface.keras', save_best_only=True)
]

# took 12 minutes
history_vggface = bmi_model_vggface.fit(
    train_gen,
    validation_data=test_gen,
    epochs=30,
    callbacks=callbacks
)

Epoch 1/30
101/101 ━━━━━━━━━━━━━━━━━━━━ 84s 756ms/step - loss: 146.7845 - mae: 9.1221 - val_loss: 128.3695 - val_mae: 8.4442
Epoch 2/30
101/101 ━━━━━━━━━━━━━━━━━━━━ 50s 496ms/step - loss: 80.9262 - mae: 6.9570 - val_loss: 148.3870 - val_mae: 9.5525
Epoch 3/30
101/101 ━━━━━━━━━━━━━━━━━━━━ 83s 820ms/step - loss: 70.0987 - mae: 6.5769 - val_loss: 89.4194 - val_mae: 6.8964
Epoch 4/30
101/101 ━━━━━━━━━━━━━━━━━━━━ 63s 622ms/step - loss: 64.8426 - mae: 6.2541 - val_loss: 72.4222 - val_mae: 6.3289
Epoch 5/30
101/101 ━━━━━━━━━━━━━━━━━━━━ 49s 488ms/step - loss: 61.5793 - mae: 6.0792 - val_loss: 93.4542 - val_mae: 7.0174
Epoch 6/30
101/101 ━━━━━━━━━━━━━━━━━━━━ 50s 492ms/step - loss: 57.8276 - mae: 5.8649 - val_loss: 97.0576 - val_mae: 7.2672
Epoch 7/30
101/101 ━━━━━━━━━━━━━━━━━━━━ 49s 484ms/step - loss: 52.0728 - mae: 5.6271 - val_loss: 76.2669 - val_mae: 6.3702
Epoch 8/30
101/101 ━━━━━━━━━━━━━━━━━━━━ 83s 828ms/step - loss: 49.2139 - mae: 5.5235 - val_loss: 71.3389 - val_mae: 6.1623
Epoch 9/30
10

Eval

In [41]:
# eval
preds_vggface = bmi_model_vggface.predict(test_gen).flatten()
actuals = test_df['bmi'].values

r, _ = pearsonr(preds_vggface, actuals)
print(f"Pearson r: {r:.4f}")

24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 279ms/step
Pearson r: 0.5169


Save kera models (skip training next time)

In [40]:
bmi_model.save('/content/drive/Shareddrives/TEAM 7 Machine Learning II/Final Project/vgg16_bmi.keras')
bmi_model_vggface.save('/content/drive/Shareddrives/TEAM 7 Machine Learning II/Final Project/vggface_bmi.keras')

# and to reload next session:
#from tensorflow.keras.models import load_model
#bmi_model = load_model('/content/drive/Shareddrives/TEAM 7 Machine Learning II/Final Project/vgg16_bmi.keras')

# Ok now we have 2 base models. Both still worse the paper's 0.65 score.

We should try to retrain the VGG-Face upgrade, but unfreeze more layers (we only did the last 4) and lower the learning race. We could also add gender as an input feature, since we have it and the paper shows a gender gap in performance.

# We can also build the API now just to get that started. Once we have a final model, swapping out which model the API is using would be a one line change.